# Explainable AI with LIME
## Explainable Fake News Detection using Transformer-based Language Models

**Student Name:** Abdullah  
**Reg Number:** M24F0044DS009  
**Course:** Natural Language Processing  

### Objective
The objective of this notebook is to apply LIME (Local Interpretable Model-agnostic Explanations) to explain individual predictions made by the fine-tuned BERT model.


In [ ]:
!pip install lime transformers torch


In [ ]:
# Import Liberaries

import torch
import numpy as np
import pandas as pd

from transformers import BertTokenizer, BertForSequenceClassification
from lime.lime_text import LimeTextExplainer

import warnings
warnings.filterwarnings("ignore")


In [ ]:
# Check if GPU is available

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
# Load Test Dataset

test_df = pd.read_csv("../data/shorttextpreprocessedtest.csv")
test_df.head()


In [ ]:
# Load Trained Model & Tokenizer

model_path = "../models/bert_fake_news_model"

tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)
model.to(device)
model.eval()


In [ ]:
# Define Prediction Function for LIME

class_names = ["Fake News", "Real News"]

def predict_proba_lime(texts):
    encodings = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=128,
        return_tensors="pt"
    )
    
    input_ids = encodings['input_ids'].to(device)
    attention_mask = encodings['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
    probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    return probs.cpu().numpy()


In [ ]:
# Initialize LIME Text Explainer

explainer = LimeTextExplainer(class_names=class_names)


In [ ]:
# Select instances to explain

# Select a random instance from test set
idx = 0
sample_text = test_df.iloc[idx]['text']
true_label = test_df.iloc[idx]['label']

print("Text to Explain:\n")
print(sample_text)
print("\nTrue Label:", class_names[true_label])


In [ ]:
# Generate LIME Explanation

exp = explainer.explain_instance(
    sample_text,
    predict_proba_lime,
    num_features=10,
    num_samples=1000
)


In [ ]:
# Visualization of Explanation

exp.show_in_notebook(text=True)


In [ ]:
# Save explanation as HTML
exp.save_to_file("../results/lime_explanation.html")


In [ ]:
fig = exp.as_pyplot_figure()
fig.savefig("../results/lime_explanation.png", dpi=300, bbox_inches="tight")


## Interpretation of LIME Results

- LIME provides instance-level explanations by highlighting the most influential words.
- The explanation confirms that the model relies on meaningful textual cues rather than random patterns.
- This improves trust and transparency for individual predictions.


## Final Conclusion

This project successfully implements a Transformer-based fake news detection system using BERT.
Explainable AI techniques, including SHAP and LIME, were applied to interpret model predictions at both global and local levels.
The results demonstrate that the system is accurate, transparent, and suitable for real-world misinformation detection tasks.
